# GMM calibration of beta_j parameters to match wealth moments
This notebook is the code for calibrating the 7 discount factor parameters $(\beta_1, \beta_2, ...\beta_7)$ to match 9 wealth moments using GMM. Let the data moment vector be a 9 x 1 vector defined as $m(b)$ where $b$ represents wealth data from the Survey of Consumer Finances (SCF),
\begin{equation}
  m(b)\equiv
      \begin{bmatrix} \phi_1 \\ \phi_2 \\ \vdots \\ \phi_7 \\ gini \\ var[\ln(b)]
      \end{bmatrix}
\end{equation}
where $\phi_j$ is the wealth share of the $j$th percentile of the population, $gini$ is the gini coefficient on wealth, and $var[\ln(b)]$ is the variance of log wealth (another measure of income inequality).

The model moment vector is a corresponding 9 x 1 vector defined as $m(b|\beta)$ where $b$ are the steady-state wealth data $\bar{b}_{j,s}$ from the model, along with any population weights needed to calculate the moments.
\begin{equation}
  m(b|\beta)\equiv
      \begin{bmatrix} \phi_1(b|\beta) \\ \phi_2(b|\beta) \\ \vdots \\ \phi_7(b|\beta) \\ gini(b|\beta) \\ var[\ln(b)](b|\beta)
      \end{bmatrix}
\end{equation}

Define a 9 x 1 error vector as the difference between the model moments $m(b|\beta)$ and data moments $m(b)$ given data $b$ and parameters $\beta$.
\begin{equation}
  e(b|\beta) \equiv m(b|\beta) - m(b)
\end{equation}
The calibration problem is to choose parameter vector $\beta = [\beta_1, \beta_2, ...\beta_7]$ to minimize a norm on the error vector $e(b|\beta)$ or a norm on the distance between the model moment vector $m(b|\beta)$ and the data moment vector $m(b)$. The most common GMM setup is to use a weighted $L^2$ norm or weighted sum of squared errors,
\begin{equation}
  \min_{\beta}\: e(b|\beta)^T \: W \: e(b|\beta)
\end{equation}
where $W$ is a 9 x 9 weighting matrix. We will initially use an identity matrix for the weighting matrix of the moments.

## Data moments

In [6]:
import numpy as np
from ogusa import wealth
from ogusa.parameters import Specifications

p = Specifications()
lambdas = p.lambdas.flatten()
scf_yrs_lst = [2019, 2016]
df_scf = wealth.get_wealth_data(scf_yrs_list=scf_yrs_lst)
mom_vec_data = wealth.compute_wealth_moments(df_scf, lambdas)
print('mom_vec_data = ')
print(mom_vec_data)

Accessing SCF data files...
Progress: |██████████████████████████████████████████████████| 100.0% Complete
Computing...
mom_vec_data = 
[-4.45113347e-03  1.77591623e-02  5.31002303e-02  5.50986027e-02
  1.10969152e-01  3.88773772e-01  3.78750214e-01  8.55996526e-01
  5.42313218e+00]


When combining data from SCF 2019 and 2016 as shown above in `mom_vec_data`, we get the following vector of wealth moments.
\begin{equation}
  m(b)\equiv
      \begin{bmatrix} \phi_1 \\ \phi_2 \\ \phi_3 \\ \phi_4 \\ \phi_5 \\ \phi_6 \\ \phi_7 \\ gini \\ var[\ln(b)]
      \end{bmatrix} = 
      \begin{bmatrix}
        -0.45\% \\ 1.78\% \\ 5.31\% \\ 5.51\% \\ 11.10\% \\ 38.88\% \\ 37.88\% \\ 0.86 \\ 5.42
      \end{bmatrix}
\end{equation}

## Model moments

### Run the steady-state solution given vector of beta_j

In [11]:
# import modules
import time
import os
import multiprocessing
from distributed import Client

import taxcalc
from taxcalc import Calculator

from ogusa import SS
from ogusa.execute import runner
from ogusa.constants import REFORM_DIR, BASELINE_DIR
from ogusa.utils import safe_read_pickle

In [28]:
client = Client()
num_workers = min(multiprocessing.cpu_count(), 7)
print('Number of workers = ', num_workers)
CUR_DIR = '/Users/richardevans/Documents/Economics/OSE/OG-USA/run_examples'

# Set some OG model parameters
# See default_parameters.json for more description of these parameters
beta_j_an0 = np.array([0.96, 0.96, 0.96, 0.96, 0.96, 0.96, 0.96])
alpha_T = np.zeros(50)  # Adjusting the path of transfer spending
alpha_T[0:2] = 0.09
alpha_T[2:10] = 0.09 + 0.01
alpha_T[10:40] = 0.09 - 0.01
alpha_T[40:] = 0.09
alpha_G = np.zeros(7)  # Adjusting the path of non-transfer spending
alpha_G[0:3] = 0.05 - 0.01
alpha_G[3:6] = 0.05 - 0.005
alpha_G[6:] = 0.05
# Set start year for baseline and reform.
START_YEAR = 2021
run_micro = False
test = False
time_path = False
baseline = True
guid0 = '_base0'
tax_func_path = os.path.join(
    CUR_DIR, '..', 'ogusa', 'data', 'tax_functions',
    'TxFuncEst_baseline_CPS.pkl')  # use cached baseline estimates
spec0 = Specifications(run_micro=run_micro, tax_func_path=tax_func_path,
                       test=test, baseline=baseline, guid=guid0,
                       client=client, num_workers=num_workers)
og_spec0 = {'frisch': 0.41, 'start_year': START_YEAR, 'cit_rate': [0.21],
            'debt_ratio_ss': 1.0, 'alpha_T': alpha_T.tolist(),
            'alpha_G': alpha_G.tolist(), 'beta_annual': beta_j_an0}
spec0.update_specifications(og_spec0)
spec0.get_tax_function_parameters(client, run_micro, tax_func_path)
start_time0 = time.time()
ss_outputs0 = SS.run_SS(spec0, client=client)
client.close()
elapsed_time0 = time.time() - start_time0
print('run time 0 = ', elapsed_time0, 'seconds')

/Users/richardevans/opt/anaconda3/envs/ogusa-dev/lib/python3.8/site-packages/distributed/node.py:151: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 57220 instead
  warnings.warn(


Number of workers =  7
Tax Function Path Exists
GE loop errors =  0.0016126379936728247 [0.01168843 0.01612548 0.01758635 0.01074636 0.01279776 0.01538276
 0.00303975] -0.0018472345395550843 0.01056223394008815
GE loop errors =  0.001612637993674379 [0.01168843 0.01612548 0.01758635 0.01074636 0.01279776 0.01538276
 0.00303975] -0.0018472345395554243 0.01056223394008865
GE loop errors =  0.001612637993674379 [0.01168843 0.01612548 0.01758635 0.01074636 0.01279776 0.01538276
 0.00303975] -0.0018472345395554243 0.010562233940088622
GE loop errors =  0.0016126366188805863 [0.01168843 0.01612548 0.01758635 0.01074636 0.01279776 0.01538276
 0.00303975] -0.0018472344781163275 0.010562233876961258
GE loop errors =  0.001612637992849844 [0.01168843 0.01612548 0.01758635 0.01074636 0.01279776 0.01538276
 0.00303975] -0.0018472345395904727 0.01056223394037839
GE loop errors =  0.0016126379928632084 [0.01168843 0.01612548 0.01758635 0.01074636 0.01279776 0.01538276
 0.00303975] -0.001847234539614

### Calculate the data moments from the steady-state output

In [40]:
from ogusa.utils import Inequality

base0_ineq = Inequality(ss_outputs0['bssmat_s'], spec0.omega_SS,
                        spec0.lambdas, spec0.S, spec0.J)
base0_ineq.gini()

0.5608841937088869

In [44]:
client = Client()

beta_j_an1 = np.array([0.90, 0.91, 0.92, 0.93, 0.94, 0.96, 0.99])
guid1 = '_base1'
spec1 = Specifications(run_micro=run_micro, tax_func_path=tax_func_path,
                       test=test, baseline=baseline, guid=guid1,
                       client=client, num_workers=num_workers)
og_spec1 = {'frisch': 0.41, 'start_year': START_YEAR, 'cit_rate': [0.21],
            'debt_ratio_ss': 1.0, 'alpha_T': alpha_T.tolist(),
            'alpha_G': alpha_G.tolist(), 'beta_annual': beta_j_an1}
spec1.update_specifications(og_spec1)
spec1.get_tax_function_parameters(client, run_micro, tax_func_path)
start_time1 = time.time()
ss_outputs1 = SS.run_SS(spec1, client=client)
client.close()
elapsed_time1 = time.time() - start_time1
print('run time 1 = ', elapsed_time1, 'seconds')

/Users/richardevans/opt/anaconda3/envs/ogusa-dev/lib/python3.8/site-packages/distributed/node.py:151: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 57494 instead
  warnings.warn(


Tax Function Path Exists
GE loop errors =  0.02180235565606002 [0.00862275 0.01236263 0.00392237 0.00926131 0.01159093 0.0156617
 0.00450164] -0.007308333187688754 0.021896398772273173
GE loop errors =  0.016342282779147357 [0.00858015 0.01230221 0.01437724 0.00921657 0.01153508 0.01558651
 0.00448012] -0.00570195861863277 0.018202840782347407
GE loop errors =  0.016342282779147385 [0.00858015 0.01230221 0.01437724 0.00921657 0.01153508 0.01558651
 0.00448012] -0.00570195861863277 0.018202840782347407
GE loop errors =  0.016342281488943386 [0.00858015 0.01230221 0.01437724 0.00921657 0.01153508 0.01558651
 0.00448012] -0.005701958583351839 0.01820284074305986
GE loop errors =  0.016342282778356018 [0.00858015 0.01230221 0.01437724 0.00921657 0.01153508 0.01558651
 0.00448012] -0.0057019586186947 0.018202840782683527
GE loop errors =  0.016342282778329012 [0.00858015 0.01230221 0.01437724 0.00921657 0.01153508 0.01558651
 0.00448012] -0.0057019586187083485 0.01820284078272874
GE loop er

In [51]:
base1_ineq = Inequality(ss_outputs1['bssmat_splus1'], spec1.omega_SS,
                        spec1.lambdas, spec1.S, spec1.J)
print('base1 Gini coef =', base1_ineq.gini())
print('base1 var_of_logs =', base1_ineq.var_of_logs())

base1 Gini coef = 0.6106301889750227
base1 var_of_logs = 1.637676671265157


In [49]:
bssmat_s = ss_outputs1['bssmat_s']
print(bssmat_s.shape, bssmat_s.min())

(80, 7) 0.0


In [53]:
np.log(bssmat_s.min())

-inf

In [52]:
spec1.omega_SS.shape

(80,)

In [56]:
bssmat_splus1 = ss_outputs1['bssmat_splus1']
bssvec_sp1 = bssmat_splus1.flatten()
wgtvec_s = spec1.omega_SS
wgtvec_j = spec1.lambdas
S = spec1.S
J = spec1.J
wgtmat_sj = (np.tile(wgtvec_s.reshape(S, 1), (1, J)) *
                np.tile(wgtvec_j.reshape(1, J), (S, 1)))
wgtvec_sj = wgtmat_sj.flatten()
index = 
print(wgtvec_sj)

[3.94979129e-03 3.94979129e-03 3.15983304e-03 1.57991652e-03
 1.57991652e-03 1.42192487e-03 1.57991652e-04 3.96900760e-03
 3.96900760e-03 3.17520608e-03 1.58760304e-03 1.58760304e-03
 1.42884274e-03 1.58760304e-04 3.99013082e-03 3.99013082e-03
 3.19210466e-03 1.59605233e-03 1.59605233e-03 1.43644710e-03
 1.59605233e-04 4.01245381e-03 4.01245381e-03 3.20996305e-03
 1.60498152e-03 1.60498152e-03 1.44448337e-03 1.60498152e-04
 4.03489081e-03 4.03489081e-03 3.22791265e-03 1.61395632e-03
 1.61395632e-03 1.45256069e-03 1.61395632e-04 4.05675806e-03
 4.05675806e-03 3.24540645e-03 1.62270322e-03 1.62270322e-03
 1.46043290e-03 1.62270322e-04 4.07671285e-03 4.07671285e-03
 3.26137028e-03 1.63068514e-03 1.63068514e-03 1.46761663e-03
 1.63068514e-04 4.09448778e-03 4.09448778e-03 3.27559022e-03
 1.63779511e-03 1.63779511e-03 1.47401560e-03 1.63779511e-04
 4.11006636e-03 4.11006636e-03 3.28805309e-03 1.64402654e-03
 1.64402654e-03 1.47962389e-03 1.64402654e-04 4.12306275e-03
 4.12306275e-03 3.298450

In [60]:
(spec1.lambdas.flatten() * beta_j_an1).sum()

0.9198000000000001

In [58]:
beta_j_an1

array([0.9 , 0.91, 0.92, 0.93, 0.94, 0.96, 0.99])

In [59]:
spec1.lambdas

array([[0.25],
       [0.25],
       [0.2 ],
       [0.1 ],
       [0.1 ],
       [0.09],
       [0.01]])